In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_file_path: str
    model_dir: Path
    best_model_path: str
    mlflow_tracking_uri: str
    experiment_name: str
    target_column: str

In [6]:
from Car_Price_Prediction_System.constants import *
from Car_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_file_path=config.test_data_file_path,
            model_dir=config.model_dir,
            best_model_path=config.best_model_path,
            mlflow_tracking_uri=config.mlflow_tracking_uri,
            experiment_name=config.experiment_name,
            target_column=schema.name
        )

        return model_evaluation_config

In [8]:
from Car_Price_Prediction_System import logger
import numpy as np
import pandas as pd
import mlflow
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate(self):
        logger.info("Loading Testing Data")
        test_df = pd.read_csv(self.config.test_data_file_path)

        X_test = test_df.drop(columns=[self.config.target_column])
        y_test = test_df[self.config.target_column]

        model = {
            "RandomForestRegressor": "RandomForestRegressor.joblib",
            "GradientBoostingRegressor": "GradientBoostingRegressor.joblib",
            "AdaBoostRegressor": "AdaBoostRegressor.joblib",
            "ExtraTreesRegressor": "ExtraTreesRegressor.joblib"
        }

        best_score = float("-inf")
        best_model = None
        best_model_name = None

        mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        mlflow.set_experiment(self.config.experiment_name)

        for model_name, filename in model.items():
            logger.info(f"Evaluating {model_name}")

            model_path = os.path.join(self.config.model_dir, filename)

            model = joblib.load(model_path)

            predictions = model.predict(X_test)

            r2 = r2_score(y_test, predictions)
            mae = mean_absolute_error(y_test, predictions)
            rmse = np.sqrt(mean_squared_error(y_test, predictions))

            logger.info(f"{model_name} -> R2: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")

            with mlflow.start_run(run_name=f"{model_name}_evaluation"):
                mlflow.log_metric("r2_score", r2)
                mlflow.log_metric("mae", mae)
                mlflow.log_metric("rmse", rmse)

                if r2 > best_score:
                    best_score = r2
                    best_model = model
                    best_model_name = model_name

            logger.info(f"Best Model: {best_model_name}")
            logger.info(f"Best R2 Score: {best_score}")

            joblib.dump(best_model, self.config.best_model_path)
            logger.info("Best Model Saved Successfully")

In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-07-23 09:14:10,861] : INFO : common : Created File: <_io.TextIOWrapper name='config/config.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 09:14:10,868] : INFO : common : Created File: <_io.TextIOWrapper name='config/params.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 09:14:10,870] : INFO : common : Created File: <_io.TextIOWrapper name='config/schema.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 09:14:10,873] : INFO : common : Created Directory: artifacts
[2026-07-23 09:14:10,875] : INFO : common : Created Directory: artifacts/model_evaluation
[2026-07-23 09:14:10,876] : INFO : 2026698328 : Loading Testing Data
[2026-07-23 09:14:14,293] : INFO : 2026698328 : Evaluating RandomForestRegressor
[2026-07-23 09:14:16,243] : INFO : 2026698328 : RandomForestRegressor -> R2: 0.9580, MAE: 0.5493, RMSE: 0.9839
[2026-07-23 09:14:16,683] : INFO : 2026698328 : Best Model: RandomForestRegressor
[2026-07-23 09:14:16,686] : INFO : 2026698328 : Best R2 Score: 0.9579741286113767
[2026-07-23 09:14:16,8